## Part 1 - Dataset & Preprocessing

In [1]:
# ================================
# Part 1 - Dataset & Preprocessing
# VS Code version: choose CSV locally, no Google Colab / Google Drive
# ================================

# ================================
# 1. Import Libraries
# ================================
from pathlib import Path
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import seaborn as sns

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)

# ================================
# 2. Choose Dataset File in VS Code
# ================================
def choose_csv_file():
    """
    VS Code local CSV chooser.
    Option 1: choose a CSV that is already inside your project folder.
    Option 2: paste the full path to any CSV file.
    Option 3: press Enter to open a file picker window.
    """
    current_folder = Path.cwd()
    csv_files = sorted(current_folder.glob("*.csv"))

    if csv_files:
        print("CSV files found in this VS Code folder:")
        for i, file in enumerate(csv_files, start=1):
            print(f"{i}. {file.name}")

        choice = input(
            "\nEnter the file number, paste a full CSV path, "
            "or press Enter to open file picker: "
        ).strip()

        if choice.isdigit():
            index = int(choice) - 1
            if 0 <= index < len(csv_files):
                return csv_files[index]

        if choice:
            return Path(choice.strip('"').strip("'"))

    # File picker fallback
    try:
        from tkinter import Tk, filedialog

        root = Tk()
        root.withdraw()
        root.attributes("-topmost", True)

        selected_file = filedialog.askopenfilename(
            title="Choose your CSV dataset",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
        )

        root.destroy()

        if selected_file:
            return Path(selected_file)

    except Exception as error:
        print(f"File picker could not open: {error}")

    manual_path = input("Paste your CSV file path here: ").strip()
    return Path(manual_path.strip('"').strip("'"))


file_path = choose_csv_file()

if not file_path.exists():
    raise FileNotFoundError(f"CSV file not found: {file_path}")

print(f"\nSelected file: {file_path.resolve()}")

# ================================
# 3. Load Dataset
# ================================
df = pd.read_csv(file_path)

print("==== Dataset Preview ====")
print(df.head())

print("\n==== Dataset Info ====")
print(df.info())

print("\n==== Missing Values ====")
print(df.isnull().sum())

# ================================
# 4. Strip whitespace and filter English reviews only
# ================================
df['userLang'] = df['userLang'].astype(str).str.strip()
df_en = df[df['userLang'] == 'EN'].copy()
print("\n==== English Reviews Count ====")
print(len(df_en))

# ================================
# 5. Map app_id to default app names
# ================================
app_mapping = {
    'org.telegram.messenger': 'Telegram',
    'com.facebook.orca': 'Facebook Messenger',
    'com.whatsapp': 'WhatsApp',
    'com.viber.voip': 'Viber',
    'com.snapchat.android': 'Snapchat',
    'com.tencent.mm': 'WeChat'
}

df_en['app_name'] = df_en['app_id'].map(app_mapping)
print("\n==== Sample App Names Mapping ====")
print(df_en[['app_id', 'app_name']].head(10))

# ================================
# 6. Map scores to sentiment
# ================================
def map_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df_en['Sentiment'] = df_en['score'].apply(map_sentiment)
print("\n==== Sentiment Distribution ====")
print(df_en['Sentiment'].value_counts())

CSV files found in this VS Code folder:
1. final_model_predictions.csv
2. preprocessed_reviews.csv
3. Training_Data_Google_Play_reviews_6000.csv

Selected file: C:\Coding space\Assignment nlp\SAIA2163-NLP-Final-Project\Training_Data_Google_Play_reviews_6000.csv
==== Dataset Preview ====
                               reviewId           userName  \
0  495266a4-f451-48c3-a844-fb3c07560d55     Foysal Hossain   
1  947fcd83-7a28-403d-b03b-d0bc20f52e0e          S K VERMA   
2  65856211-67ba-4560-84dd-a0055775ed90      Amanuel Abara   
3  cd5ba250-3a26-43b4-a378-77d18f73a503  Vagarangas X Aopi   
4  e8e886b4-d6c6-416b-b0a1-be90320c4024       Shafin islam   

                                           userImage  \
0  https://play-lh.googleusercontent.com/a-/ALV-U...   
1  https://play-lh.googleusercontent.com/a/ACg8oc...   
2  https://play-lh.googleusercontent.com/a/ACg8oc...   
3  https://play-lh.googleusercontent.com/a/ACg8oc...   
4  https://play-lh.googleusercontent.com/a-/ALV-U...   

  

In [2]:
# ================================
# 6️. Convert Review Dates to Datetime
# ================================
df_en['review_date'] = pd.to_datetime(df_en['at'], errors='coerce')
print("\n==== Sample Review Dates ====")
print(df_en[['at', 'review_date']].head(10))

# ================================
# 7️. Preprocessing Function with Stemming
# ================================
nltk.download('punkt_tab') # Add this line to download the missing resource
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

def preprocess_text_stem(text):
    """Preprocess text with stemming"""
    if pd.isnull(text):
        return ''
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove punctuation, emojis, and numbers but keep normal words
    # Important: the old regex deleted the words by mistake.
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenization
    words = nltk.word_tokenize(text)
    # Remove stopwords and apply stemming
    words = [ps.stem(w) for w in words if w not in stop_words and len(w) > 1]
    return ' '.join(words)

# ================================
# 8️ Apply Preprocessing
# ================================
df_en['clean_text'] = df_en['content'].apply(preprocess_text_stem)
print("\n==== Sample Preprocessed Reviews ====")
print(df_en[['content', 'clean_text', 'Sentiment', 'app_name']].head(10))

# ================================
# 9️. Save Preprocessed CSV
# ================================
df_final = df_en[['clean_text', 'Sentiment', 'app_name', 'review_date']]\
    .rename(columns={'clean_text':'text', 'Sentiment':'label'})

df_final.to_csv("preprocessed_reviews.csv", index=False)
print("\n==== Preprocessed dataset saved as 'preprocessed_reviews.csv' ====")


==== Sample Review Dates ====
                    at         review_date
0  2023-09-19 15:05:31 2023-09-19 15:05:31
1  2023-09-19 14:59:30 2023-09-19 14:59:30
2  2023-09-19 14:55:06 2023-09-19 14:55:06
3  2023-09-19 14:50:18 2023-09-19 14:50:18
4  2023-09-19 14:48:20 2023-09-19 14:48:20
5  2023-09-19 14:46:54 2023-09-19 14:46:54
6  2023-09-19 14:43:18 2023-09-19 14:43:18
7  2023-09-19 14:42:49 2023-09-19 14:42:49
8  2023-09-19 14:37:57 2023-09-19 14:37:57
9  2023-09-19 14:37:46 2023-09-19 14:37:46

==== Sample Preprocessed Reviews ====
                              content                          clean_text  \
0          Gett van for no reason 😂😂😂                     gett van reason   
1               better' than WhatsApp                     better whatsapp   
2            That was good app for me                            good app   
3                        Love the app                            love app   
4                              🕳️🕳️🕳️                                   

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Part **2**

In [3]:
# ================================================
# Part 2 - Model Training, Evaluation & Backend Export
# ================================================

# ================================================
# 1️. Import Required Libraries
# ================================================
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.pipeline import Pipeline

# Libraries for Transformers
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# ================================================
# 2️. Load and Prepare Preprocessed Data
# ================================================
print("==== Loading Dataset ====")
df_final = pd.read_csv("preprocessed_reviews.csv")

# Handle any missing/empty text values gracefully to avoid vectorizer crashes
df_final['text'] = df_final['text'].fillna('missing_text')
df_final['text'] = df_final['text'].replace('', 'missing_text')

# Map textual labels to integers for classification
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
df_final['target'] = df_final['label'].map(label_mapping)

# Run train-test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    df_final['text'],
    df_final['target'],
    test_size=0.2,
    random_state=42,
    stratify=df_final['target']
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}\n")

# ================================================
# 3️. Feature Extraction (TF-IDF Vectorizer)
# ================================================
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ================================================
# 4️. Model Training & Evaluation (Traditional ML)
# ================================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB()
}

results = {}
best_acc = 0
best_model_name = None
best_model_obj = None

for name, model in models.items():
    # Train model
    model.fit(X_train_tfidf, y_train)
    # Predict
    preds = model.predict(X_test_tfidf)

    # Calculate Metrics
    acc = accuracy_score(y_test, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted', zero_division=0)
    cm = confusion_matrix(y_test, preds)

    results[name] = {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "confusion_matrix": cm,
        "model": model
    }

    print(f"==== {name} Performance ====")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}\n")

    # Track the absolute best traditional model
    if acc > best_acc:
        best_acc = acc
        best_model_name = name
        best_model_obj = model

# ================================================
# 5️. Export Backend for Streamlit Deployment
# ================================================
print(f"🥇 Absolute Best Traditional Model: {best_model_name} ({best_acc:.4f} Accuracy)")

best_pipeline = Pipeline([
    ('tfidf', vectorizer),
    ('classifier', best_model_obj)
])

joblib.dump(best_pipeline, 'best_sentiment_pipeline.pkl')
print("Pipeline saved successfully as 'best_sentiment_pipeline.pkl'")

backend_metadata = {
    "best_model_name": best_model_name,
    "metrics": {
        "accuracy": results[best_model_name]["accuracy"],
        "precision": results[best_model_name]["precision"],
        "recall": results[best_model_name]["recall"],
        "f1_score": results[best_model_name]["f1_score"],
    },
    "confusion_matrix": results[best_model_name]["confusion_matrix"].tolist(),
    "label_mapping": label_mapping
}
joblib.dump(backend_metadata, 'backend_metadata.pkl')
print("Backend metadata saved successfully as 'backend_metadata.pkl'\n")


# ================================================
# Advanced NLP Model (BERT / Transformers)
# ================================================
print("================================================")
print("Training DistilBERT Transformer Model")
print("================================================")

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using execution device: {device}\n")

# Reformat data split for Hugging Face Dataset format
hf_train_df = pd.DataFrame({'text': X_train, 'label': y_train}).reset_index(drop=True)
hf_test_df = pd.DataFrame({'text': X_test, 'label': y_test}).reset_index(drop=True)

train_dataset = Dataset.from_pandas(hf_train_df)
test_dataset = Dataset.from_pandas(hf_test_df)

# Load pre-trained Tokenizer
transformer_model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(transformer_model_name)

# Tokenization utility function
def tokenize_inputs(data):
    return tokenizer(data["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_inputs, batched=True)
tokenized_test = test_dataset.map(tokenize_inputs, batched=True)

# Initialize Model with 3 distinct output classes
transformer_model = AutoModelForSequenceClassification.from_pretrained(transformer_model_name, num_labels=3)

# Define hardware-optimized training configuration
training_args = TrainingArguments(
    output_dir="./transformer_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    report_to="none"
)

# Metric aggregation for Transformer evaluation
def compute_transformer_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

# Initialize the Hugging Face Trainer
trainer = Trainer(
    model=transformer_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_transformer_metrics,
)

# Begin fine-tuning
print("Fine-tuning DistilBERT")
trainer.train()

# Evaluate the transformer performance
print("\n==== Evaluating DistilBERT Performance ====")
eval_metrics = trainer.evaluate()
print(f"DistilBERT Accuracy:  {eval_metrics['eval_accuracy']:.4f}")
print(f"DistilBERT Precision: {eval_metrics['eval_precision']:.4f}")
print(f"DistilBERT Recall:    {eval_metrics['eval_recall']:.4f}")
print(f"DistilBERT F1-Score:  {eval_metrics['eval_f1']:.4f}")

==== Loading Dataset ====
Training set size: 960
Testing set size: 240

==== Logistic Regression Performance ====
Accuracy:  0.7333
Precision: 0.6876
Recall:    0.7333
F1-Score:  0.7086

==== Naive Bayes Performance ====
Accuracy:  0.7000
Precision: 0.6579
Recall:    0.7000
F1-Score:  0.6668

🥇 Absolute Best Traditional Model: Logistic Regression (0.7333 Accuracy)
Pipeline saved successfully as 'best_sentiment_pipeline.pkl'
Backend metadata saved successfully as 'backend_metadata.pkl'

Training DistilBERT Transformer Model
Using execution device: cpu



Map:   0%|          | 0/960 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Coding space\Assignment nlp\SAIA2163-NLP-Final-Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned mem

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


==== Evaluating DistilBERT Performance ====


c:\Coding space\Assignment nlp\SAIA2163-NLP-Final-Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.626274,0.707524,2,0.704167,0.657868,0.704167,0.678333


DistilBERT Accuracy:  0.7042
DistilBERT Precision: 0.6579
DistilBERT Recall:    0.7042
DistilBERT F1-Score:  0.6783


In [4]:
import nltk

print("Downloading Training Dependencies")

# Core NLTK resources for tokenization and text alignment
nltk.download('punkt')
nltk.download('punkt_tab')

print("All NLTK resources downloaded")

All NLTK resources downloaded


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
from pathlib import Path
import os

print("==== Generating Final Predictions CSV ====")

results_df = pd.DataFrame({
    'clean_text': X_test,
    'true_target': y_test
})

inverse_label_mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}
results_df['true_sentiment'] = results_df['true_target'].map(inverse_label_mapping)

for name, model in models.items():
    safe_name = name.lower().replace(" ", "_")
    results_df[f'pred_{safe_name}'] = model.predict(X_test_tfidf)
    results_df[f'sentiment_{safe_name}'] = results_df[f'pred_{safe_name}'].map(inverse_label_mapping)

final_export_df = results_df.drop(columns=['true_target', 'pred_logistic_regression', 'pred_naive_bayes'])

# VS Code version: save output in the current project folder
final_export_path = Path.cwd() / "final_model_predictions.csv"
final_export_df.to_csv(final_export_path, index=False)

print("Saved to:")
print(f" -> {final_export_path.resolve()}")
print("\nPreview of Saved Product")
print(final_export_df.head())

==== Generating Final Predictions CSV ====
Saved to:
 -> C:\Coding space\Assignment nlp\SAIA2163-NLP-Final-Project\final_model_predictions.csv

Preview of Saved Product
                                             clean_text true_sentiment  \
1188  hi team wechat use realm scan anoth phone redm...       negative   
898                                                mast       positive   
276                          non sens unabl send messag       negative   
1040  dear builder creat new account tri sign need s...       negative   
554                                       whtup respond       negative   

     sentiment_logistic_regression sentiment_naive_bayes  
1188                      negative              negative  
898                       positive              positive  
276                       negative              negative  
1040                      negative              negative  
554                       negative              negative  
